In [7]:
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
import json


In [8]:
links = pd.read_csv("Data/links_germany50.csv")

nodes = sorted(set(links["source"]).union(set(links["target"])))
node_to_id = {name: i for i, name in enumerate(nodes)}
id_to_node = {i: name for name, i in node_to_id.items()}

len(node_to_id)

50

In [9]:
with open("RSA_estimation/germany50/node_mapping.json", "w") as f:
    json.dump(node_to_id, f, indent=2)

In [10]:
import networkx as nx

G = nx.Graph()
for _, r in links.iterrows():
    u = node_to_id[r.source]
    v = node_to_id[r.target]
    G.add_edge(u, v)

In [11]:
G.number_of_nodes(), G.number_of_edges()

(50, 88)

In [12]:
import numpy as np

rng = np.random.default_rng(42)

N_REQUESTS = 1000
bitrate_choices = [50, 100, 200, 400]

requests = []

for _ in range(N_REQUESTS):
    s, d = rng.choice(list(node_to_id.values()), size=2, replace=False)
    b = rng.choice(bitrate_choices)
    requests.append((s, d, b))

df_requests = pd.DataFrame(
    requests,
    columns=["source", "destination", "bitrate"]
)

In [13]:
req_dir = Path("RSA_estimation/germany50/request-set_0")
req_dir.mkdir(parents=True, exist_ok=True)
df_requests.to_csv(req_dir / "requests.csv", index=False)

## Validate node mapping + traffic consistency

In [14]:
import pandas as pd
import json
from pathlib import Path

In [15]:
links = pd.read_csv("Data/links_germany50.csv")

nodes = sorted(set(links["source"]).union(set(links["target"])))
node_to_id = {name: i for i, name in enumerate(nodes)}
id_to_node = {i: name for name, i in node_to_id.items()}

print(f"Number of nodes: {len(node_to_id)}")

Number of nodes: 50


In [16]:
list(node_to_id.items())[:10]

[('Aachen', 0),
 ('Augsburg', 1),
 ('Bayreuth', 2),
 ('Berlin', 3),
 ('Bielefeld', 4),
 ('Braunschweig', 5),
 ('Bremen', 6),
 ('Bremerhaven', 7),
 ('Chemnitz', 8),
 ('Darmstadt', 9)]

In [17]:
req_dir = Path("RSA_estimation/germany50/request-set_0")
requests = pd.read_csv(req_dir / "requests.csv")
requests.head()

,source,destination,bitrate
0,4,38,100
1,42,21,200
2,9,4,400
3,36,38,400
4,25,6,100


In [18]:
assert requests["source"].dtype.kind in "iu"
assert requests["destination"].dtype.kind in "iu"

assert requests["source"].min() >= 0
assert requests["destination"].min() >= 0

assert requests["source"].max() < len(node_to_id)
assert requests["destination"].max() < len(node_to_id)

print("✔ Traffic node IDs are valid")

✔ Traffic node IDs are valid


In [19]:
mapping_path = Path("RSA_estimation/germany50/node_mapping.json")
with open(mapping_path, "w") as f:
    json.dump(node_to_id, f, indent=2)

print(f"✔ Node mapping saved to {mapping_path}")

✔ Node mapping saved to RSA_estimation\germany50\node_mapping.json


In [20]:
import networkx as nx
import numpy as np
import math

In [21]:
G = nx.Graph()
for _, r in links.iterrows():
    u = node_to_id[r.source]
    v = node_to_id[r.target]
    G.add_edge(u, v)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph: 50 nodes, 88 edges


In [22]:
SLOT_WIDTH_GHZ = 12.5
MAX_SLOTS = 4000

# Example modulation model (ADJUST later if needed)
SPECTRAL_EFFICIENCY = 4.0  # bits/s/Hz


In [23]:
def slots_required(bitrate_gbps: float) -> int:
    bandwidth = bitrate_gbps / SPECTRAL_EFFICIENCY
    return math.ceil(bandwidth / SLOT_WIDTH_GHZ)

In [24]:
def select_path(G, src: int, dst: int):
    return nx.shortest_path(G, src, dst)

In [25]:
def first_fit_allocate(occupancy, path, slots_needed):
    for start in range(MAX_SLOTS - slots_needed):
        feasible = True
        for u, v in zip(path[:-1], path[1:]):
            key = edge_key(u, v)
            if any(occupancy[key][start:start + slots_needed]):
                feasible = False
                break
        if feasible:
            for u, v in zip(path[:-1], path[1:]):
                key = edge_key(u, v)
                occupancy[key][start:start + slots_needed] = [1] * slots_needed
            return start + slots_needed
    raise RuntimeError("Spectrum exhausted")

In [26]:
req_dir = Path("RSA_estimation/germany50/request-set_0")
requests = pd.read_csv(req_dir / "requests.csv")

len(requests), requests.head()

(1000,
    source  destination  bitrate
 0       4           38      100
 1      42           21      200
 2       9            4      400
 3      36           38      400
 4      25            6      100)

In [27]:
occupancy = {
    (u, v): [0] * MAX_SLOTS
    for u, v in G.edges()
}

In [28]:
def edge_key(u, v):
    return (u, v) if u < v else (v, u)

occupancy = {
    edge_key(u, v): [0] * MAX_SLOTS
    for u, v in G.edges()
}

In [29]:
highest_slots = []
active_transceivers = []

for idx, req in requests.iterrows():
    src = int(req.source)
    dst = int(req.destination)
    bitrate = float(req.bitrate)

    path = select_path(G, src, dst)
    slots = slots_required(bitrate)

    h = first_fit_allocate(occupancy, path, slots)

    highest_slots.append(h)
    active_transceivers.append(1)

    if idx % 100 == 0:
        print(f"Processed request {idx}")

Processed request 0
Processed request 100
Processed request 200
Processed request 300
Processed request 400
Processed request 500
Processed request 600
Processed request 700
Processed request 800
Processed request 900


In [30]:
metrics = {
    "highestSlot": max(highest_slots),
    "avgHighestSlot": float(np.mean(highest_slots)),
    "sumOfSlots": int(sum(highest_slots)),
    "avgActiveTransceivers": float(np.mean(active_transceivers)),
}

metrics

{'highestSlot': 756,
 'avgHighestSlot': 164.659,
 'sumOfSlots': 164659,
 'avgActiveTransceivers': 1.0}

In [31]:
with open(req_dir / "results.txt", "w") as f:
    for k, v in metrics.items():
        f.write(f"{k}\t{v}\n")

print("✔ results.txt written for request-set_0")

✔ results.txt written for request-set_0


In [32]:
def run_rsa_batch(G, root_dir):
    for req_dir in sorted(root_dir.glob("request-set_*")):
        req_csv = req_dir / "requests.csv"
        out_txt = req_dir / "results.txt"

        if not req_csv.exists():
            continue

        print(f"[RSA] Processing {req_dir.name}")

        requests = pd.read_csv(req_csv)

        occupancy = {
            edge_key(u, v): [0] * MAX_SLOTS
            for u, v in G.edges()
        }

        highest_slots = []
        active_transceivers = []

        for _, req in requests.iterrows():
            src = int(req.source)
            dst = int(req.destination)
            bitrate = float(req.bitrate)

            path = select_path(G, src, dst)
            slots = slots_required(bitrate)

            h = first_fit_allocate(occupancy, path, slots)

            highest_slots.append(h)
            active_transceivers.append(1)

        metrics = {
            "highestSlot": max(highest_slots),
            "avgHighestSlot": float(np.mean(highest_slots)),
            "sumOfSlots": int(sum(highest_slots)),
            "avgActiveTransceivers": float(np.mean(active_transceivers)),
        }

        with open(out_txt, "w") as f:
            for k, v in metrics.items():
                f.write(f"{k}\t{v}\n")

        print(f"  ✔ results.txt written")

In [33]:
run_rsa_batch(G, Path("RSA_estimation/germany50"))

[RSA] Processing request-set_0
  ✔ results.txt written


In [34]:
import numpy as np
import pandas as pd
from pathlib import Path

In [35]:
def generate_request_set(
    node_ids,
    n_requests=1000,
    bitrate_choices=(50, 100, 200, 400),
    seed=None,
):
    rng = np.random.default_rng(seed)
    requests = []

    for _ in range(n_requests):
        src, dst = rng.choice(node_ids, size=2, replace=False)
        bitrate = rng.choice(bitrate_choices)
        requests.append((src, dst, bitrate))

    return pd.DataFrame(
        requests,
        columns=["source", "destination", "bitrate"]
    )

In [36]:
ROOT = Path("RSA_estimation/germany50")
NODE_IDS = list(node_to_id.values())

N_SETS = 100        # start with 20
N_REQUESTS = 1000 # per set

In [37]:
for i in range(N_SETS):
    req_dir = ROOT / f"request-set_{i}"
    req_dir.mkdir(parents=True, exist_ok=True)

    df_req = generate_request_set(
        node_ids=NODE_IDS,
        n_requests=N_REQUESTS,
        seed=42 + i   # different seed per set
    )

    df_req.to_csv(req_dir / "requests.csv", index=False)

    print(f"✔ Generated request-set_{i}")

✔ Generated request-set_0
✔ Generated request-set_1
✔ Generated request-set_2
✔ Generated request-set_3
✔ Generated request-set_4
✔ Generated request-set_5
✔ Generated request-set_6
✔ Generated request-set_7
✔ Generated request-set_8
✔ Generated request-set_9
✔ Generated request-set_10
✔ Generated request-set_11
✔ Generated request-set_12
✔ Generated request-set_13
✔ Generated request-set_14
✔ Generated request-set_15
✔ Generated request-set_16
✔ Generated request-set_17
✔ Generated request-set_18
✔ Generated request-set_19
✔ Generated request-set_20
✔ Generated request-set_21
✔ Generated request-set_22
✔ Generated request-set_23
✔ Generated request-set_24
✔ Generated request-set_25
✔ Generated request-set_26
✔ Generated request-set_27
✔ Generated request-set_28
✔ Generated request-set_29
✔ Generated request-set_30
✔ Generated request-set_31
✔ Generated request-set_32
✔ Generated request-set_33
✔ Generated request-set_34
✔ Generated request-set_35
✔ Generated request-set_36
✔ Generated

In [38]:
run_rsa_batch(G, ROOT)

[RSA] Processing request-set_0
  ✔ results.txt written
[RSA] Processing request-set_1
  ✔ results.txt written
[RSA] Processing request-set_10
  ✔ results.txt written
[RSA] Processing request-set_11
  ✔ results.txt written
[RSA] Processing request-set_12
  ✔ results.txt written
[RSA] Processing request-set_13
  ✔ results.txt written
[RSA] Processing request-set_14
  ✔ results.txt written
[RSA] Processing request-set_15
  ✔ results.txt written
[RSA] Processing request-set_16
  ✔ results.txt written
[RSA] Processing request-set_17
  ✔ results.txt written
[RSA] Processing request-set_18
  ✔ results.txt written
[RSA] Processing request-set_19
  ✔ results.txt written
[RSA] Processing request-set_2
  ✔ results.txt written
[RSA] Processing request-set_20
  ✔ results.txt written
[RSA] Processing request-set_21
  ✔ results.txt written
[RSA] Processing request-set_22
  ✔ results.txt written
[RSA] Processing request-set_23
  ✔ results.txt written
[RSA] Processing request-set_24
  ✔ results.txt wri

In [39]:
import os

sum(1 for _ in ROOT.glob("request-set_*/results.txt"))

100